> ⚠️ **시작하기 전에 — 클러스터를 연결하세요**
>
> 이 랩에는 **Photon이 비활성화된 클래식 컴퓨팅 클러스터**가 필요합니다. 서버리스 컴퓨팅은 이 연습에서 사용하는 Spark 구성 옵션을 지원하지 않으며, Photon은 이 랩이 드러내도록 설계된 데이터 편향(skew) 및 셔플 문제를 자동으로 최적화하여 제거합니다. 둘 중 하나라도 활성화되어 있으면 Spark UI에서 비정상적인 내용을 전혀 볼 수 없습니다.
>
> 1. 이 노트북 상단의 컴퓨팅 선택기에서 **Connect**를 클릭하세요.
> 2. **`perf-lab`** 클러스터를 선택하세요(랩 설정 지침에서 생성됨 — 비-Photon 런타임, Spark 구성에 `spark.databricks.photon.enabled false` 설정됨).
> 3. 셀을 실행하기 전에 클러스터 상태가 **Running**으로 표시될 때까지 기다리세요.

## 연습 1: 환경 설정

성능 문제를 조사하기 전에 작업할 데이터가 필요합니다. 이 연습에서는 Unity Catalog 구조를 만들고 두 개의 Delta 테이블을 생성합니다.

- **`perf_lab.data.transactions`** — 의도적으로 편향된 `customer_id` 열을 가진 500,000개의 행(약 70 %의 행이 단일 고객에 속함)
- **`perf_lab.data.customers`** — 200개의 행(작은 차원 테이블)

계속하기 전에 이 연습의 모든 셀을 위에서 아래로 실행하세요.

### 작업 1.1: Unity Catalog 구조 만들기

이 셀은 제공됩니다 — 실행하여 카탈로그와 스키마를 만드세요.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로 기본 카탈로그 루트의 스토리지 경로를 상속합니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

In [ ]:
# ✅ 이 셀을 실행하세요 — 카탈로그와 스키마를 만듭니다
spark.sql(f"CREATE CATALOG IF NOT EXISTS perf_lab MANAGED LOCATION '{storage_root}' COMMENT '성능 문제 해결 랩'")
spark.sql("CREATE SCHEMA IF NOT EXISTS perf_lab.data COMMENT '성능 연습용 트랜잭션 데이터'")
print("✅ 카탈로그와 스키마가 준비되었습니다.")

### 작업 1.2: transactions 테이블 생성

데이터셋은 전적으로 Spark에서 생성되며 외부 파일이 필요하지 않습니다.

**의도적인 편향:** `rand(seed=42) < 0.70`은 전체 행의 약 70 %를 `customer_001`로 보냅니다. 나머지 모든 행은 `customer_002`부터 `customer_200`까지 분산됩니다. 이 불균형의 결과는 연습 2에서 확인하게 됩니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 심하게 편향된 customer_id를 가진 500,000개의 트랜잭션을 생성합니다
from pyspark.sql import functions as F

n_rows = 500_000

df_transactions = (
    spark.range(n_rows)
    .withColumn(
        "customer_id",
        F.when(F.rand(seed=42) < 0.70, F.lit("customer_001"))
         .otherwise(
             F.concat(
                 F.lit("customer_"),
                 F.lpad((F.floor(F.rand(seed=99) * 199) + 2).cast("string"), 3, "0")
             )
         )
    )
    .withColumn("amount",   F.round(F.rand(seed=7) * 999 + 1, 2))
    .withColumn("txn_type", F.when(F.rand(seed=3) < 0.5, F.lit("purchase")).otherwise(F.lit("refund")))
    .withColumn("txn_id",   F.concat(F.lit("txn_"), F.col("id").cast("string")))
    .drop("id")
)

df_transactions.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.transactions")
count = spark.table("perf_lab.data.transactions").count()
print(f"✅ {count:,}개의 행을 perf_lab.data.transactions에 기록했습니다")

### 작업 1.3: customers 테이블 생성

이 작은 차원 테이블에는 200개의 행이 있으며, 고객당 한 행입니다. 단일 실행기(executor)의 메모리에 충분히 들어갈 만큼 작습니다. 이 사실을 기억해 두세요. 연습 3에서 중요해집니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 200개의 고객 차원 행을 생성합니다
customer_rows = [
    (f"customer_{i:03d}", f"Customer {i}", f"region_{(i % 5) + 1}")
    for i in range(1, 201)
]

df_customers = spark.createDataFrame(customer_rows, ["customer_id", "name", "region"])
df_customers.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.customers")
print(f"✅ {df_customers.count()}개의 행을 perf_lab.data.customers에 기록했습니다")

### 작업 1.4: 편향 확인

아래 셀을 실행하여 데이터가 심하게 불균형한지 확인하세요. 출력에는 트랜잭션 수 기준 상위 고객과 전체 데이터셋에서 차지하는 비율이 표시됩니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — customer_id 값에 걸친 행 분포를 보여줍니다
df = spark.table("perf_lab.data.transactions")
total = df.count()

top_customers = (
    df.groupBy("customer_id")
      .agg(F.count("*").alias("txn_count"))
      .withColumn("pct_of_total", F.round(F.col("txn_count") / total * 100, 1))
      .orderBy(F.desc("txn_count"))
      .limit(5)
)

print(f"전체 행 수: {total:,}")
print("트랜잭션 수 기준 상위 5개 고객:")
top_customers.show()

---

## 연습 2: 데이터 편향 드러내기

Spark UI에서 편향을 명확하게 보려면 **적응형 쿼리 실행(AQE)**을 꺼야 합니다. AQE가 활성화되어 있으면 Databricks가 런타임에 과도하게 큰 파티션을 자동으로 분할하는데, 이는 편향을 완화하지만 동시에 UI에서 문제를 숨기기도 합니다.

`customer_id`에서 셔플을 수행하는 두 가지 작업을 실행합니다.
1. **groupBy 집계** — 셔플이 전체 행의 약 70 %를 단일 리듀서 태스크로 보냅니다.
2. transactions와 customers 간의 **sort-merge 조인** — 자동 브로드캐스트가 비활성화되어 Spark가 양쪽을 모두 셔플해야 합니다.

아래 셀을 실행한 후 **랩 지침 페이지로 돌아가서** **Part 2**의 단계에 따라 Spark UI에서 결과를 조사하세요.

### 작업 2.1: AQE 비활성화 및 셔플 파티션 설정

In [ ]:
# ✅ 이 셀을 실행하세요 — Spark UI에서 편향이 완전히 보이도록 AQE와 자동 브로드캐스트를 비활성화합니다
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled",                 "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "-1")    # sort-merge 조인 강제 적용
spark.conf.set("spark.sql.shuffle.partitions",               "50")

print("AQE 비활성화 | 자동 브로드캐스트 비활성화 | 셔플 파티션: 50")

### 작업 2.2: 편향된 집계 실행

`groupBy("customer_id")`는 셔플을 트리거합니다. `customer_001`에 약 350,000개의 행이 있기 때문에 하나의 파티션(따라서 하나의 태스크)이 전체 데이터의 약 70 %를 받습니다. 나머지 모든 파티션이 남은 30 %를 나눠 가집니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — customer_id에서 편향된 셔플을 트리거합니다
import time

df_tx = spark.table("perf_lab.data.transactions")

start = time.time()
result_agg = (
    df_tx
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
)
result_agg.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.agg_skewed")
elapsed = time.time() - start
print(f"✅ 집계 완료: {elapsed:.1f}s — {spark.table('perf_lab.data.agg_skewed').count()}개 행")

### 작업 2.3: 편향된 sort-merge 조인 실행

자동 브로드캐스트가 비활성화되어 있으므로(`autoBroadcastJoinThreshold = -1`) Spark는 조인의 **양쪽**을 모두 셔플하고 sort-merge 전략을 사용해야 합니다. 편향된 `customer_id` 분포는 하나의 리듀서 태스크가 transactions 쪽에서 불균형하게 큰 파티션을 받게 됨을 의미합니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 편향된 sort-merge 조인을 트리거합니다
df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_join = (
    df_tx
    .join(df_cust, "customer_id", "left")
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_join.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_skewed")
elapsed = time.time() - start
print(f"✅ sort-merge 조인 완료: {elapsed:.1f}s — {spark.table('perf_lab.data.join_skewed').count()}개 행")

> **이제 랩 지침 페이지로 돌아가서 Part 2의 단계에 따라 Spark UI를 조사하세요.**
>
> 조사를 마치면 아래 연습 3을 계속 진행하세요.

---

## 연습 3: 데이터 편향 해결

편향을 해결하기 위한 두 가지 상호 보완적인 도구가 있습니다.

| Technique | How it works | Best when |
|-----------|-------------|----------|
| **Broadcast join** | Sends the small table to every executor — no shuffle of either side | One side of a join is small |
| **AQE skew join** | Splits oversized shuffle partitions dynamically at runtime | General-purpose skew in joins and aggregations |

이 연습에서는 두 가지를 모두 적용합니다.

### ✅ 작업 3.1 (정답): 브로드캐스트 힌트를 사용하여 조인 수정

`df_cust`를 `F.broadcast()`로 감싸면 Spark가 전체 테이블을 모든 실행기(executor)로 복사하도록 지시합니다. customers 쪽의 셔플이 발생하지 않으며, 조인이 더 이상 `customer_id`로 데이터를 재분배하지 않기 때문에 transactions 쪽의 편향된 파티션이 태스크 지속 시간에 영향을 주지 않습니다.

In [ ]:
# ✅ 정답 — 브로드캐스트 조인은 customers 쪽의 sort-merge 셔플을 제거합니다
import time
from pyspark.sql import functions as F

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_broadcast = (
    df_tx
    .join(F.broadcast(df_cust), "customer_id", "left")  # 브로드캐스트: sort-merge 셔플 없음
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_broadcast.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_broadcast")
elapsed = time.time() - start
print(f"✅ 브로드캐스트 조인 완료: {elapsed:.1f}s — {spark.table('perf_lab.data.join_broadcast').count()}개 행")
print("이 시간을 작업 2.3의 sort-merge 조인과 비교하세요.")

### 작업 3.2: AQE 다시 활성화 및 자동 편향 처리 관찰

`spark.sql.adaptive.skewJoin.enabled = true`로 설정하면 AQE가 셔플 중에 편향된 파티션을 자동으로 감지하여 더 작은 하위 태스크로 분할합니다. 이는 코드를 전혀 변경하지 않고도 편향을 처리하며, 테이블을 브로드캐스트할 수 없거나 원하지 않을 때 유용합니다.

아래 셀은 AQE를 다시 활성화하고 작업 2.3의 원래 sort-merge 조인을 다시 실행합니다(브로드캐스트 힌트 없음). 실행 시간을 비교하고 Spark UI를 확인하여 AQE가 편향된 파티션을 어떻게 분할하는지 살펴보세요.

In [ ]:
# ✅ 이 셀을 실행하세요 — AQE를 다시 활성화하고 sort-merge 조인을 다시 실행합니다
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled",                 "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",        "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "10485760")  # 10 MB로 재설정

print("AQE 활성화 | skewJoin 활성화 | 자동 브로드캐스트 임계값: 10 MB")

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_aqe = (
    df_tx
    .join(df_cust, "customer_id", "left")
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_aqe.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_aqe")
elapsed = time.time() - start
print(f"✅ AQE 조인 완료: {elapsed:.1f}s — {spark.table('perf_lab.data.join_aqe').count()}개 행")
print("\nSpark UI를 확인하세요 — AQE가 편향된 파티션을 여러 하위 태스크로 분할합니다.")

---

## 연습 4: 과도한 셔플 드러내기

셔플은 Spark가 데이터를 재분배해야 할 때(예: `groupBy` 또는 `join`) 불가피합니다. 그러나 이미 셔플이 포함된 작업의 앞이나 뒤에 `repartition()`을 호출하면 *불필요한* 셔플을 쉽게 발생시킬 수 있습니다.

이 연습의 파이프라인은 `groupBy` 다음에 `join`과 `sort`를 수행하지만, 각각 아무 이점 없이 전체 셔플을 트리거하는 **세 개의 추가 `repartition()` 호출**을 더합니다. Spark UI는 DAG에서 여러 개의 **Exchange** 노드를 보여주는데, 이는 실제 계산에 필요한 것보다 많습니다.

아래 셀을 실행한 후 **랩 지침 페이지로 돌아가서** **Part 4**의 단계에 따라 셔플 지표를 조사하세요.

### 작업 4.1: 구성 재설정 및 셔플 집약적 파이프라인 실행

In [ ]:
# ✅ 이 셀을 실행하세요 — 모든 셔플이 보이도록 AQE와 자동 브로드캐스트를 비활성화합니다
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled",                 "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "-1")
spark.conf.set("spark.sql.shuffle.partitions",               "200")

print("AQE 비활성화 | 자동 브로드캐스트 비활성화 | 셔플 파티션: 200")

In [ ]:
# ✅ 이 셀을 실행하세요 — 불필요한 repartition이 포함된 셔플 집약적 파이프라인
#
# 셔플 1: repartition(100, "txn_type")     — groupBy 이전의 사전 repartition (불필요)
# 셔플 2: groupBy 집계              — customer_id에 대한 필요한 셔플
# 셔플 3: repartition(200, "customer_id")  — 조인 이전의 재셔플 (불필요)
# 셔플 4: customers와의 sort-merge 조인   — 브로드캐스트가 비활성화되어 셔플 발생
# 셔플 5: repartition(50)                  — 조인 이후 repartition (불필요)
# 셔플 6: orderBy                          — 필요한 전역 정렬

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_inefficient = (
    df_tx
    .repartition(100, "txn_type")                       # 셔플 1: 불필요
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
    .repartition(200, "customer_id")                    # 셔플 3: 불필요
    .join(df_cust, "customer_id", "left")               # 셔플 4: sort-merge (브로드캐스트 비활성화)
    .repartition(50)                                    # 셔플 5: 불필요
    .orderBy("region", "txn_type")                      # 셔플 6: 필요
)
result_inefficient.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.results_inefficient")
elapsed = time.time() - start
print(f"✅ 비효율적인 파이프라인 완료: {elapsed:.1f}s")
print(f"   기록된 행 수: {spark.table('perf_lab.data.results_inefficient').count():,}")
print("\n이제 랩 지침 페이지로 돌아가서 Part 4에 대한 Spark UI를 조사하세요.")

> **랩 지침 페이지로 돌아가서 Part 4의 단계에 따라 셔플 지표를 조사하세요.**
>
> 조사를 마치면 아래 연습 5를 계속 진행하세요.

---

## 연습 5: 셔플 오버헤드 줄이기

연습 4의 파이프라인은 `groupBy → join → sort`를 수행합니다. 이 계산은 근본적으로 **두 번**의 셔플이 필요합니다. 하나는 집계용이고 다른 하나는 전역 정렬용입니다. 나머지는 모두 오버헤드입니다.

### ✅ 정답: 최적화된 파이프라인

최적화된 버전은 다음과 같습니다.
1. 세 개의 `repartition()` 호출을 모두 제거합니다.
2. `F.broadcast(df_cust)`를 사용하여 sort-merge 셔플을 제거합니다.
3. 추가적인 동적 최적화를 위해 AQE를 다시 활성화합니다.
4. (정렬된 출력에 필요한) `orderBy`는 유지합니다.

이렇게 하면 `groupBy` 집계와 `orderBy`라는 **두 번**의 셔플만 남으며, 이는 피할 수 없는 최소값입니다.

In [ ]:
# ✅ 정답 — 최적화된 파이프라인: 불필요한 repartition 없음, 브로드캐스트 조인, AQE 활성화

# AQE와 브로드캐스트 임계값을 다시 활성화
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled",                 "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "10485760")  # 10 MB

print("AQE 활성화 | 자동 브로드캐스트 임계값: 10 MB")

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_optimised = (
    df_tx
    # groupBy 이전에 repartition 없음 — Spark가 셔플을 내부적으로 처리
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
    # 브로드캐스트 조인: customers가 모든 실행기로 복사됨 — sort-merge 셔플 없음
    .join(F.broadcast(df_cust), "customer_id", "left")
    # orderBy 이전에 repartition 없음
    .orderBy("region", "txn_type")                      # 남은 셔플 하나 (전역 정렬)
)
result_optimised.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.results_optimised")
elapsed = time.time() - start
print(f"✅ 최적화된 파이프라인 완료: {elapsed:.1f}s")
print(f"   기록된 행 수: {spark.table('perf_lab.data.results_optimised').count():,}")
print("\nSpark UI에서 비교하세요: Exchange 노드가 더 적고 단계별 Shuffle Read/Write가 더 적습니다.")

---

## 정리 (선택 사항)

작업을 마친 후 아래 셀을 실행하여 Unity Catalog에서 모든 랩 리소스를 제거하세요.

In [ ]:
# 주석을 해제하고 실행하여 모든 랩 리소스를 제거하세요
# spark.sql("DROP CATALOG IF EXISTS perf_lab CASCADE")
# print("✅ 랩 리소스가 정리되었습니다")